# Hash Generation
Reads every **active** row in `hash_config`, hashes each configured source table, and writes the results to the `hashed_values` output table.

The logic mirrors the Oracle procedure **`P_HEALIX_HASH_EXPORT_MAIN_V3_PIV_NL`** (the CR/LF-fix version), so `hash_value` is identical to the procedure's `ROW_MD5`.

For every configured table:
- **primary_key** = the procedure's `ID_PK` text: the value wrapped in double quotes, e.g. `"123"`
- **composite_keys** = the procedure's `PK_VAL` columns: each value wrapped in double quotes, joined with `|`, e.g. `"123"|"ABC"`
- **primary_key_hash_value / composite_keys_hash_value** = hash of the two key texts above (the Oracle procedure does not hash keys; these are Databricks-side helpers)
- **hash_value** = hash of the concatenated (no delimiter) normalized values of *every* column in `information_schema.columns` order, **excluding any column ending in `___c`** (ingestion metadata). This must be the same column set and order as the Oracle table.
- the hash algorithm is whatever `hash_algorithm` says for that table. The Oracle procedure only uses `md5`, so use `md5` for any comparison with `ROW_MD5`.

### Column normalization (procedure section 4)
| Oracle type | Text that goes into the hash |
|---|---|
| `DATE` | `TO_CHAR(c,'YYYY-MM-DD HH24:MI:SS')`, NULL -> blank (no escaping) |
| `TIMESTAMP*` | `TO_CHAR(c,'YYYY-MM-DD HH24:MI:SS.FF')`, NULL -> blank (no escaping) |
| `CLOB` | first 32,767 characters, then cleanup |
| `VARCHAR2` / `CHAR` | value, NULL -> blank, then cleanup |
| `NUMBER` / other | `TO_CHAR(c)` (Oracle minimal format: `1.00`->`1`, `0.5`->`.5`), NULL -> blank, then cleanup |

**Cleanup, in this order:** remove `CHR(13)`, remove `CHR(10)`, `"` -> `""`, remove `CHR(0)`.

Values are concatenated with **no delimiter**, hashed as UTF-8 bytes, and written as lowercase hex (same as `LOWER(RAWTOHEX(DBMS_CRYPTO.HASH(...,2)))`).

Keys use the same `TO_CHAR` text but **without** the CR/LF removal, exactly as the procedure builds `ID_PK` / `PK_VAL`.


## Imports

In [0]:
%run ./00_config

## Read active config rows

In [0]:
config_rows = (
    spark.table(config_table)
    .filter(F.col("is_active") == 1)
    .collect()
)

print(f"Found {len(config_rows)} active table(s) to process")

## Helper functions
Spark equivalents of the procedure's expressions (see the header docstring in the next cell). Three settings can be adjusted:
- `TS_FRACTION_DIGITS` (default 6) must equal the precision of the Oracle `TIMESTAMP` columns
- `ORACLE_DATE_COLUMNS` lists Oracle `DATE` columns that arrived in Databricks as `TIMESTAMP`
- the Spark session time zone must be the one the timestamps were loaded with (`spark.sql.session.timeZone`)

In [0]:
"""
Mirrors P_HEALIX_HASH_EXPORT_MAIN_V3_PIV_NL (Oracle) so ROW_MD5 == hash_value.

Per column, in column_id / ordinal_position order, concatenated with NO delimiter:
  DATE            NVL(TO_CHAR(c,'YYYY-MM-DD HH24:MI:SS'),'')
  TIMESTAMP*      NVL(TO_CHAR(c,'YYYY-MM-DD HH24:MI:SS.FF'),'')
  CLOB            DBMS_LOB.SUBSTR(NVL(c,EMPTY_CLOB()),32767,1)  -> then cleanup
  VARCHAR2/CHAR   NVL(c,'')                                      -> then cleanup
  everything else NVL(TO_CHAR(c),'')                             -> then cleanup
  cleanup order:  REPLACE CHR(13) -> '' ; CHR(10) -> '' ; '"' -> '""' ; CHR(0) -> ''
Hash = LOWER(RAWTOHEX(DBMS_CRYPTO.HASH(UTL_RAW.CAST_TO_RAW(expr), 2)))   (2 = MD5, UTF-8 bytes)
"""

# DBMS_LOB.SUBSTR(value, 32767, 1) equivalent for CLOB values
CLOB_CHAR_LIMIT = 32767

# Oracle TO_CHAR(ts, '...FF') with no digit prints the column's own precision.
# TIMESTAMP(6) (Oracle default, and Spark's precision) -> 6 digits.
TS_FRACTION_DIGITS = 6

# Oracle DATE columns that landed in Databricks as TIMESTAMP must still be hashed
# as 'YYYY-MM-DD HH24:MI:SS' (no fraction). List them here, keys lower-case:
#   ORACLE_DATE_COLUMNS = {"moh_raz.raz_casemix_raw.my_table": {"created_date", "updated_date"}}
ORACLE_DATE_COLUMNS = {}


def hash_expr(algorithm, expr):
    """Return a hash column expression for the given algorithm (lower-case hex)."""
    algorithm = algorithm.strip().lower()
    if algorithm == "md5":
        return F.md5(expr)          # Oracle DBMS_CRYPTO.HASH(..., 2)
    elif algorithm in ("sha256", "sha-256"):
        return F.sha2(expr, 256)    # not used by the Oracle procedure (MD5 only)
    else:
        raise ValueError(f"Unsupported hash_algorithm: {algorithm}")


def get_hashable_columns(catalog, schema, table):
    """(column_name, data_type) in ordinal order, excluding ingestion columns ending in ___c."""
    cols_df = spark.sql(f"""
        SELECT column_name, data_type
        FROM {catalog.lower()}.information_schema.columns
        WHERE table_schema = '{schema.lower()}'
          AND table_name   = '{table.lower()}'
          AND column_name NOT RLIKE '___c$'
        ORDER BY ordinal_position
    """)
    cols = [(r["column_name"], r["data_type"]) for r in cols_df.collect()]
    print(f"  {catalog}.{schema}.{table} columns/dtypes: {cols}")
    return cols


def _oracle_number_to_char(s):
    """
    Oracle TO_CHAR(number) with no format model gives the minimal representation:
        1.00 -> '1'     0.50 -> '.5'     -0.50 -> '-.5'     0.00 -> '0'     100 -> '100'
    Spark's cast(decimal as string) keeps the scale ('1.00', '0.50'), so normalise it.
    """
    has_fraction = s.contains(".") & ~s.contains("E")
    trimmed = F.regexp_replace(F.regexp_replace(s, r"0+$", ""), r"\.$", "")
    s = F.when(has_fraction, trimmed).otherwise(s)
    return F.regexp_replace(s, r"^(-?)0\.", "$1.")


def _to_char_base(colname, data_type, date_cols=frozenset()):
    """
    TO_CHAR equivalent per datatype. Result is never NULL (NULL -> ''), and has NO
    CR/LF/quote/NUL cleanup yet.
    """
    dtype = (data_type or "").strip().lower()
    col = F.col(colname)

    if dtype == "date" or colname.lower() in date_cols:
        return F.coalesce(F.date_format(col, "yyyy-MM-dd HH:mm:ss"), F.lit(""))

    if dtype.startswith("timestamp"):
        fmt = "yyyy-MM-dd HH:mm:ss." + "S" * TS_FRACTION_DIGITS
        return F.coalesce(F.date_format(col, fmt), F.lit(""))

    if dtype.startswith(("decimal", "numeric", "dec", "float", "double", "real")):
        return F.coalesce(_oracle_number_to_char(col.cast("string")), F.lit(""))

    if dtype == "binary":                       # Oracle TO_CHAR(RAW) -> upper-case hex
        return F.coalesce(F.upper(F.hex(col)), F.lit(""))

    return F.coalesce(col.cast("string"), F.lit(""))


def standardize_column(colname, data_type, date_cols=frozenset()):
    """One column of the ROW_MD5 expression (procedure section 4)."""
    s = _to_char_base(colname, data_type, date_cols)
    s = F.substring(s, 1, CLOB_CHAR_LIMIT)       # CLOB: DBMS_LOB.SUBSTR(...,32767,1)
    s = F.regexp_replace(s, r"\r", "")           # CHR(13)
    s = F.regexp_replace(s, r"\n", "")           # CHR(10)
    s = F.regexp_replace(s, '"', '""')           # " -> ""
    s = F.regexp_replace(s, r"\x00", "")         # CHR(0)
    return s


def concat_columns(col_specs, date_cols=frozenset()):
    """ROW_MD5 input: all columns, no delimiter."""
    return F.concat(*[standardize_column(n, t, date_cols) for n, t in col_specs])


def key_value_expr(col_specs, date_cols=frozenset()):
    """
    Key text exactly as the procedure writes ID_PK / PK_VAL (sections 5-6):
        each value  = '"' || REPLACE(REPLACE(NVL(TO_CHAR(c),''),'"','""'),CHR(0),'') || '"'
        several     = joined with '|'          (NO CR/LF removal for keys)
        no key cols = '""'
    """
    if not col_specs:
        return F.lit('""')
    parts = []
    for name, dtype in col_specs:
        v = _to_char_base(name, dtype, date_cols)
        v = F.regexp_replace(v, '"', '""')
        v = F.regexp_replace(v, r"\x00", "")
        parts.append(F.concat(F.lit('"'), v, F.lit('"')))
    return F.concat_ws("|", *parts)

## Process each configured table

In [0]:
output_table = f"{tgt_catalog}.{tgt_schema}.{tgt_table}"

def split_cols(csv_text):
    return [c.strip() for c in (csv_text or "").split(",") if c.strip()]

for row in config_rows:
    src_catalog    = row["catalog"]
    src_schema     = row["schema"]
    src_table      = row["table"]
    primary_key    = split_cols(row["primary_key"])      # procedure: ID_PK (0 or 1 column)
    composite_keys = split_cols(row["composite_keys"])   # procedure: PK_VAL columns
    hash_algorithm = row["hash_algorithm"].strip().lower()

    full_table_name = f"{src_catalog}.{src_schema}.{src_table}"
    print(f"Processing {full_table_name} using {hash_algorithm}...")

    all_columns = get_hashable_columns(src_catalog, src_schema, src_table)
    if not all_columns:
        print(f"  WARNING: no hashable columns found for {full_table_name}, skipping")
        continue

    # config uses upper-case names (CASE_SURR_ID), information_schema is lower-case:
    # look up case-insensitively and fail loudly instead of silently treating a key as STRING
    col_types = {name.lower(): dtype for name, dtype in all_columns}
    def key_specs(names):
        missing = [n for n in names if n.lower() not in col_types]
        if missing:
            raise ValueError(f"{full_table_name}: key column(s) {missing} not found in hashable columns")
        return [(n, col_types[n.lower()]) for n in names]

    date_cols = {c.lower() for c in ORACLE_DATE_COLUMNS.get(full_table_name.lower(), set())}

    src_df = spark.table(full_table_name)

    pk_value_expr  = key_value_expr(key_specs(primary_key), date_cols)
    ck_value_expr  = key_value_expr(key_specs(composite_keys), date_cols)
    row_value_expr = concat_columns(all_columns, date_cols)

    result_df = src_df.select(
        F.lit(src_catalog).alias("catalog"),
        F.lit(src_schema).alias("schema"),
        F.lit(src_table).alias("table"),
        pk_value_expr.alias("primary_key"),
        hash_expr(hash_algorithm, pk_value_expr).alias("primary_key_hash_value"),
        ck_value_expr.alias("composite_keys"),
        hash_expr(hash_algorithm, ck_value_expr).alias("composite_keys_hash_value"),
        hash_expr(hash_algorithm, row_value_expr).alias("hash_value"),
        F.lit(hash_algorithm).alias("hash_algo"),
        F.current_timestamp().alias("loaded_at___c"),
    )

    # Temp - Refresh (rather than duplicate) this table rows in the output table
    spark.sql(f"""
        DELETE FROM {output_table}
        WHERE catalog = '{src_catalog}'
          AND schema  = '{src_schema}'
          AND table   = '{src_table}'
    """)

    (
        result_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(output_table)
    )

    row_count = result_df.count()
    print(f"  Appended {row_count} row(s) from {full_table_name} to {output_table}\n")

print("All configured tables processed.")

## Sanity check

In [0]:
display(spark.table(f"{tgt_catalog}.{tgt_schema}.{tgt_table}").limit(50))